In [ ]:
!git clone https://github.com/Scofe-C/Generative_Project_1.git

## Section 2 — Configuration & Initialization
Load the master config, resolve device, and validate all pipeline parameters before touching any data.

In [ ]:
import sys, os
os.chdir('/content/Generative_Project_1')
sys.path.insert(0, '/content/Generative_Project_1')
print(os.getcwd())

In [ ]:
# config.yaml is the single source of truth for ALL parameters.
# Never hardcode dataset names, batch sizes, or class lists here.

from src.utils import load_config, setup_logging, set_seed, get_device
from src.utils import get_per_class_target, get_candidate_pool_size, get_pipeline_mode

# Load config from the cloned repo
config = load_config('configs/config.yaml')

# Initialize logging (writes to logs/pipeline.log + console)
setup_logging(config)

# Set all random seeds for reproducibility
# This covers: Python random, NumPy, PyTorch CPU, PyTorch CUDA
set_seed(config['dataset']['seed'])

print('Config loaded successfully')
print(f"Pipeline mode   : {get_pipeline_mode(config)}")
print(f"Dataset         : {config['dataset']['name']}")
print(f"Split           : {config['dataset']['split']}")
print(f"Target N        : {config['dataset']['target_n']}")
print(f"Classes ({len(config['dataset']['classes'])})     : {config['dataset']['classes']}")
print(f"Per-class target: {get_per_class_target(config)}")
print(f"Candidate pool  : {get_candidate_pool_size(config)} (= target × {config['dataset']['pool_multiplier']}x)")
print(f"Seed            : {config['dataset']['seed']}")

In [ ]:
# utils.get_device() checks CUDA → MPS → CPU in order.
# On Colab Pro with A100, this always returns cuda.

import torch
from src.utils import get_device, get_pin_memory

device = get_device()

print(f'Device          : {device}')
print(f'Pin memory      : {get_pin_memory(device, config)}')
print(f'Batch size      : {config["hardware"]["batch_size"]}')

if device.type == 'cuda':
    # Log available VRAM before any models are loaded
    free_vram  = torch.cuda.mem_get_info(0)[0] / 1e9
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM free       : {free_vram:.1f} / {total_vram:.1f} GB')

# Guard: batch_size > 64 risks OOM on A100 for CLIP+GPT-2 simultaneously
# Reduce to 32 in config.yaml if you see CUDA OOM errors
assert config['hardware']['batch_size'] <= 64, (
    f"batch_size={config['hardware']['batch_size']} may OOM. "
    "Reduce to 32 or less in config.yaml."
)

In [ ]:
# Quick dry-run: verify HuggingFace Hub can find both model configs
# before we start downloading COCO data.
# This fails fast if model names are wrong or Hub is unreachable.

from transformers import AutoConfig

clip_name = config['models']['clip']
gpt2_name = config['models']['gpt2']

print(f'Checking CLIP model  : {clip_name}')
clip_cfg = AutoConfig.from_pretrained(clip_name)
print(f'  → model_type       : {clip_cfg.model_type}')
print(f'  → projection_dim   : {clip_cfg.projection_dim}')

print(f'Checking GPT-2 model : {gpt2_name}')
gpt2_cfg = AutoConfig.from_pretrained(gpt2_name)
print(f'  → model_type       : {gpt2_cfg.model_type}')
print(f'  → hidden_size      : {gpt2_cfg.n_embd}')
print(f'  → vocab_size       : {gpt2_cfg.vocab_size}')

# Store dims for use in later sections
CLIP_EMBED_DIM = clip_cfg.projection_dim  # 512 for ViT-B/32
GPT2_HIDDEN_DIM = gpt2_cfg.n_embd         # 768 for gpt2-base

print(f'\nEmbedding dimensions confirmed:')
print(f'  CLIP image embedding : {CLIP_EMBED_DIM}')
print(f'  GPT-2 hidden size    : {GPT2_HIDDEN_DIM}')
print('\n Model configs resolved — safe to proceed')

In [ ]:
# Ensure outputs/ logs/ and checkpoints/ exist before the pipeline runs.
# These are .gitignored but must exist at runtime.

from pathlib import Path
from src.utils import get_output_path

# Resolve output path from config (creates outputs/ if it doesn't exist)
output_path = get_output_path(config)

# Create logs/ directory
log_dir = Path(config['logging']['log_dir'])
log_dir.mkdir(parents=True, exist_ok=True)

# Create checkpoints/ directory for M3
Path('checkpoints').mkdir(exist_ok=True)

print(f'Output path   : {output_path}')
print(f'Log dir       : {log_dir}')
print(f'Checkpoint dir: checkpoints/')
print('\n All output directories ready')

In [ ]:
# Override config classes with Flickr30k-abundant categories
# Based on scan: person, man, woman, dog, water, street, car, bike, child, girl
# all have 300+ matches in Flickr30k
config['dataset']['classes'] = [
    'man', 'woman', 'dog', 'child', 'car',
    'street', 'water', 'girl', 'bike', 'horse'
]
config['dataset']['target_n'] = 1000

# Update alias map so data_loader can match these
from src.data_loader import COCO_CLASS_ALIASES
COCO_CLASS_ALIASES.update({
    'man':    ['man'],
    'woman':  ['woman'],
    'child':  ['child'],
    'girl':   ['girl'],
    'street': ['street'],
    'water':  ['water'],
    'bike':   ['bike'],
    'horse':  ['horse'],
    'dog':    ['dog'],
    'car':    ['car'],
})

print(f"Classes updated: {config['dataset']['classes']}")
print(f"Per-class target: {get_per_class_target(config)}")

In [ ]:
# Target: 300 candidates per class (3x the 100 per-class target).
# Expected runtime: 5–10 minutes depending on network speed.

import logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.WARNING)
logging.getLogger("filelock").setLevel(logging.WARNING)

from src.data_loader import build_candidate_pool

print('Building candidate pool — streaming COCO (this may take 5–10 minutes)...')
print(f'Fetching {get_candidate_pool_size(config)} candidates across {len(config["dataset"]["classes"])} classes\n')

buckets = build_candidate_pool(config)

# Show per-class counts
print('\nCandidate pool summary:')
total = 0
for cls, records in buckets.items():
    print(f'  {cls:<12}: {len(records):>4} candidates')
    total += len(records)
print(f'  {"TOTAL":<12}: {total:>4}')

In [ ]:
from src.data_loader import get_dataloader_splits, flatten_candidates
from src.preprocessor import Preprocessor

# Select per-class target from pool
selected = get_dataloader_splits(buckets, config)
flat = flatten_candidates(selected)
print(f"Selected {len(flat)} candidates for preprocessing")

# Preprocess
preprocessor = Preprocessor(config)
result = preprocessor.process_candidates(flat, reserve_pool=buckets)
print(f"\n{result.summary()}")

In [ ]:
import torch
import numpy as np
from transformers import AutoModel
from src.embeddings_io import EmbeddingCheckpointer, load_embeddings
from src.pipeline import _normalize, _pool_and_project
from src.utils import get_output_path

batch_size = config['hardware']['batch_size']
valid_samples = result.valid_samples

# Load models
print("Loading CLIP model...")
clip_model = AutoModel.from_pretrained(config['models']['clip']).to(device)
clip_model.eval()

print("Loading GPT-2 model...")
gpt2_model = AutoModel.from_pretrained(config['models']['gpt2']).to(device)
gpt2_model.eval()

# Create projection layer once (768 → 512)
text_projection = None
if gpt2_model.config.n_embd != clip_model.config.projection_dim:
    text_projection = torch.nn.Linear(gpt2_model.config.n_embd, clip_model.config.projection_dim, bias=False).to(device)
    torch.nn.init.kaiming_uniform_(text_projection.weight)
    text_projection.weight.requires_grad_(False)
    text_projection.eval()

# Extract embeddings
checkpointer = EmbeddingCheckpointer(config)

print(f"\nExtracting embeddings: {len(valid_samples)} samples, batch_size={batch_size}")
with torch.no_grad():
    for i in range(0, len(valid_samples), batch_size):
        batch = valid_samples[i : i + batch_size]

        pixel_values = torch.stack([s["pixel_values"] for s in batch]).to(device)
        token_ids = torch.stack([s["token_ids"] for s in batch]).to(device)
        ids = [s["image_id"] for s in batch]
        labels = [s["label"] for s in batch]

        image_emb = _normalize(clip_model.get_image_features(pixel_values=pixel_values))

        text_hidden = gpt2_model(input_ids=token_ids).last_hidden_state
        text_emb = text_hidden.mean(dim=1)
        if text_projection is not None:
            text_emb = text_projection(text_emb)
        text_emb = _normalize(text_emb)

        checkpointer.add_batch(
            image_embeddings=image_emb.cpu().numpy(),
            text_embeddings=text_emb.cpu().numpy(),
            ids=ids, labels=labels,
        )

        if (i // batch_size) % 5 == 0:
            print(f"  Batch {i//batch_size + 1}/{(len(valid_samples) + batch_size - 1)//batch_size} | {checkpointer.accumulated_count} samples")

        if device.type == 'cuda':
            torch.cuda.empty_cache()

output_path = checkpointer.finalize()
print(f"\nEmbeddings saved to: {output_path}")
print(f"Total samples: {checkpointer.accumulated_count}")

In [ ]:
data = load_embeddings(output_path, config)

print(f"Loaded {len(data['ids'])} samples")
print(f"Image embeddings shape: {data['image_embeddings'].shape}")
print(f"Text embeddings shape:  {data['text_embeddings'].shape}")

print("\nPer-class distribution:")
unique, counts = np.unique(data['labels'], return_counts=True)
for cls, count in zip(unique, counts):
    print(f"  {cls:<12}: {count}")

In [ ]:
import matplotlib.pyplot as plt

# Pick 5 random samples across different classes
rng = np.random.RandomState(42)
indices = rng.choice(len(valid_samples), size=5, replace=False)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, idx in zip(axes, indices):
    sample = valid_samples[idx]
    img_emb = data['image_embeddings'][idx]
    txt_emb = data['text_embeddings'][idx]

    # Cosine similarity
    cos_sim = np.dot(img_emb, txt_emb) / (np.linalg.norm(img_emb) * np.linalg.norm(txt_emb))

    ax.imshow(sample.get('original_image', sample['pixel_values'].permute(1,2,0).numpy() * 0.5 + 0.5))
    ax.set_title(f"{sample['label']}\ncos={cos_sim:.3f}", fontsize=9)
    ax.axis('off')

plt.suptitle("5 Sample Test Runs — Image + CLIP/GPT-2 Cosine Similarity", fontsize=12)
plt.tight_layout()
plt.savefig('outputs/sample_test_runs.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to outputs/sample_test_runs.png")